# ConvNeXt-SE for multiclass Mpox skin-lesion classification (MSLD v2.0)

This notebook runs the full workflow of the paper step by step, using the functions in `convnext_se_mpox.py` (keep both files in the same folder).

1. Setup and data loading
2. Repeated stratified 5-fold cross-validation (Tables 6–7)
3. Held-out test evaluation with TTA and ROC-AUC (Tables 8, 9, 11)
4. Ablation study (Table 10)
5. Grad-CAM

## 1. Setup

In [ ]:
!pip install -q grad-cam
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, json
from pathlib import Path
sys.path.append('.')   # folder containing convnext_se_mpox.py
import convnext_se_mpox as m

DATA = '/content/drive/MyDrive/FOLDS/Fold_1'   # MSLD v2.0 original images
OUT  = Path('outputs'); OUT.mkdir(exist_ok=True)
SEEDS = [42, 43, 44]

m.set_seed(m.CFG['global_seed'])
print('Device:', m.DEVICE)
print(json.dumps(m.CFG, indent=2))

## 2. Data loading (original images, exact duplicates removed)

In [ ]:
paths, labels, class_names = m.load_dataset(DATA)
class_names

## 3. Repeated stratified 5-fold cross-validation (3 × 5 = 15 folds)
Produces Table 6 (mean ± std, 95% CI), Table 7 (per class) and the per-fold macro-F1 plot.

In [ ]:
cv_summary = m.run_cv(paths, labels, class_names, OUT)
cv_summary

## 4. Held-out test evaluation (70/15/15) with six-view TTA and ROC-AUC
Produces Tables 8, 9 and 11, the confusion matrix, ROC curves and the trained weights for each seed.

In [ ]:
results = []
model = test_set = None
for s in SEEDS:
    mdl, ts, res = m.run_holdout(paths, labels, class_names, OUT, s)
    results.append(res)
    if model is None:
        model, test_set = mdl, ts
    else:
        del mdl

In [ ]:
import numpy as np, pandas as pd
rows = {}
for part in ('standard', 'tta'):
    for k in ('accuracy', 'precision', 'recall', 'f1', 'macro_auc'):
        v = [r[part][k] for r in results]
        rows[(part, k)] = f"{np.mean(v):.2f} ± {np.std(v, ddof=1):.2f}"
pd.Series(rows).unstack(0)

## 5. Cumulative ablation study (Table 10)

In [ ]:
ablation = m.run_ablation(paths, labels, class_names, OUT, SEEDS)
ablation

## 6. Grad-CAM explanations

In [ ]:
m.run_gradcam(model, test_set, class_names, OUT)
from IPython.display import Image
Image(str(OUT / 'gradcam_examples.png'))